# 2 — Interrogare i dati

Ricette pronte da copiare. Tutte partono dai dati caricati con `lib.loaders`,
che applicano già le pulizie necessarie (righe malformate scartate, date
convertite, tipi coerenti).

Se preferisci il CSV grezzo senza pulizie, usa il notebook 1.

In [ ]:
# Setup: rende importabile lib/ da qualunque cartella si lanci il notebook
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "lib").is_dir():          # risale finché non trova la radice del progetto
    if ROOT.parent == ROOT:                 # arrivati a "/": meglio un errore chiaro
        raise RuntimeError("lanciare il notebook da dentro tennis_project/")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from lib import explore, loaders, catalog

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

print("radice:", ROOT)

## I match ufficiali

200mila match ATP dal 1968. `load_tml()` legge i CSV grezzi; se hai già
costruito i dataset derivati (`python3 -m lib.build --all`), `load_processed`
ti dà lo stesso indice **più** la colonna `charted`, che dice se esiste
l'annotazione colpo per colpo.

In [ ]:
m = loaders.load_tml("atp")
print(f"{len(m):,} match, dal {m.tourney_date.min().date()} al {m.tourney_date.max().date()}")
m.head(3)

In [ ]:
# Versione arricchita (richiede: python3 -m lib.build --all)
try:
    mm = loaders.load_processed("matches")
    print(f"{len(mm):,} match | annotati colpo per colpo: {int(mm.charted.sum()):,}")
except FileNotFoundError as e:
    print(e)

## Ricetta: gli scontri diretti tra due giocatori

In [ ]:
def h2h(df, a, b):
    """Tutti gli incontri tra due giocatori, in ordine di data.

    Il confronto passa da normalize_name perché le fonti scrivono i nomi in
    modo diverso (Auger Aliassime / Auger-Aliassime, Oconnell / O'Connell).
    """
    n = loaders.normalize_name
    a, b = n(a), n(b)
    w, l = n(df.winner_name), n(df.loser_name)
    hit = df[((w == a) & (l == b)) | ((w == b) & (l == a))]
    return hit.sort_values("tourney_date")

x = h2h(m, "Carlos Alcaraz", "Jannik Sinner")
print(f"{len(x)} incontri — {x.winner_name.value_counts().to_dict()}")
x[["tourney_date", "tourney_name", "surface", "round", "winner_name", "score"]]

## Ricetta: tutti i match di un giocatore in una stagione

In [ ]:
GIOCATORE = "Jannik Sinner"
ANNO = 2025

n = loaders.normalize_name
target = n(GIOCATORE)
sub = m[((n(m.winner_name) == target) | (n(m.loser_name) == target))
        & (m.tourney_date.dt.year == ANNO)]

print(f"{len(sub)} match nel {ANNO} — vittorie: {(n(sub.winner_name) == target).sum()}")
sub[["tourney_date", "tourney_name", "surface", "round", "winner_name", "loser_name", "score"]]

## Ricetta: statistiche per giocatore

`tml_to_long()` trasforma una riga per match in due righe (una per giocatore),
e `add_serve_metrics()` aggiunge le percentuali con i denominatori giusti.

In [ ]:
long = loaders.add_serve_metrics(loaders.tml_to_long(m[m.tourney_date.dt.year >= 2023]))
print(f"{len(long):,} righe giocatore-match")

top = (long.groupby("player")
           .agg(match=("serve_pts", "size"), punti=("serve_pts", "sum"),
                ace=("aces", "sum"), servizio=("serve_pts_won_pct", "mean"),
                risposta=("return_pts_won_pct", "mean"))
           .query("match >= 80"))
top["ace_pct"] = (top.ace / top.punti).round(3)
top.sort_values("ace_pct", ascending=False).head(10).round(3)

## Ricetta: confronto per superficie

In [ ]:
(long.groupby("surface")
     .agg(match=("serve_pts", "size"),
          prime_in=("first_in_pct", "mean"),
          punti_al_servizio=("serve_pts_won_pct", "mean"),
          ace=("ace_pct", "mean"))
     .round(3))

## Ricetta: il dettaglio colpo per colpo di un match

Solo per i match annotati dal Match Charting Project. Si parte dall'indice,
si trova il `match_id`, e con quello si entra nei file di statistiche.

In [ ]:
mcp = loaders.load_mcp_matches("m")

# I due nomi possono stare in player_1 o player_2 indifferentemente: il
# confronto va fatto sull'insieme della coppia, non colonna per colonna.
COPPIA = {"Carlos Alcaraz", "Jannik Sinner"}
scelta = mcp[mcp.apply(lambda r: {r.player_1, r.player_2} == COPPIA, axis=1)].sort_values("date")

print(f"{len(scelta)} match annotati")
scelta[["match_id", "date", "tournament", "round", "surface"]].tail(3)

In [ ]:
MATCH_ID = scelta.match_id.iloc[-1]
print(MATCH_ID)

ov = loaders.load_mcp_stats("Overview", "m")
loaders.add_serve_metrics(ov[ov.match_id == MATCH_ID])[
    ["player", "serve_pts", "first_in_pct", "first_won_pct",
     "serve_pts_won_pct", "winners", "unforced"]].round(3)

In [ ]:
# Lunghezza degli scambi in quel match — ricordarsi il filtro su row
rally = explore.read("charting-m-stats-Rally.csv")
rally[(rally.match_id == MATCH_ID) & rally.row.isin(["1-3", "4-6", "7-9", "10"])]